---
## IMPORTS
---

In [ ]:
from __future__ import annotations

import json
import re
from collections.abc import Callable, Mapping, Sequence
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from statistics import mean
from textwrap import dedent
from typing import Any, Final, NotRequired, Required, TypedDict

from anthropic.types import Message, MessageParam, TextBlock

from learning_journey.claude.client import get_client

In [ ]:
class LLMParams(TypedDict, total=False):
    model: Required[str]
    max_tokens: Required[int]
    messages: Required[list[MessageParam]]
    system: NotRequired[str]
    stop_sequences: NotRequired[list[str]]

---
## CLIENT INITIALIZATION AND HELPER FUNCTIONS
---

In [ ]:
client, settings = get_client()


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    *,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
) -> str:
    params: LLMParams = {
        "model": settings.claude_model,
        "max_tokens": settings.default_max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    reponse: Message = client.messages.create(**params)
    text = "".join(block.text for block in reponse.content if isinstance(block, TextBlock))
    return text

---
## REPORT BUILDER
---

In [ ]:
def generate_prompt_evaluation_report(evaluation_results: Sequence[Mapping[str, Any]]) -> str:
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0

    max_possible_score: Final[int] = 10

    pass_rate = 100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap;
                word-wrap: break-word;
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        match score:
            case score if score >= 8:
                score_class = "score-high"
            case score if score <= 5:
                score_class = "score-low"
            case _:
                score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

---
## PromptEvaluator IMPLEMENTATION
---

In [ ]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks: int = 3) -> None:
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string: str, variables: Mapping[str, Any]) -> str:
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace("{" + placeholder + "}", str(variables[placeholder]))

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(
        self, task_description: str, prompt_inputs_spec: Mapping[str, str], num_cases: int
    ) -> dict[str, str]:
        """Generate a single test case based on the task description and a specific idea"""
        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:

        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>

        Each idea should represent a distinct scenario or example that tests different
        aspects of the task.

        Output Format:
        Provide your response as a structured JSON array where each item is a brief
        description of the idea.

        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```

        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas

        """

        system_prompt = """You are a test scenario designer specialized
        in creating diverse, unique testing scenarios.
        """

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs_spec": example_prompt_inputs,
            },
        )

        messages: list[MessageParam] = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
        )
        return json.loads(text)

    def generate_test_case(
        self, task_description: str, idea: str, prompt_inputs_spec: Mapping[str, str]
    ) -> dict[str, str]:
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:

        <task_description>
        {task_description}
        </task_description>

        <specific_idea>
        {idea}
        </specific_idea>

        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>

        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for
            evaluating the solution, 1 to 4 items
        }}
        ```

        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description
          and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage
        about renewable energy that covers solar power economics, wind turbine technology, and
        policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "The transition to renewable energy encompasses numerous interdependent
                dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with
                panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%,
                making it economically competitive with fossil fuels in many markets. Concurrently,
                wind energy has evolved through innovative turbine designs featuring carbon-fiber
                composite blades and advanced control systems that increase energy capture by 35% in
                low-wind conditions."
            },
            "solution_criteria": [
                "Includes all topics mentioned"
            ]
        }
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything
        outside of the scope of the task description.
        """

        system_prompt = """
        You are a test case creator specializing in designing evaluation scenarios.
        """

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages: list[MessageParam] = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea
        return test_case

    def generate_dataset(
        self,
        task_description: str,
        prompt_inputs_spec: Mapping[str, str] = {},
        num_cases: int = 1,
        output_file: str = "dataset.json",
    ) -> list[dict[str, str]]:
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(task_description, prompt_inputs_spec, num_cases)

        dataset: list[dict[str, str]] = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        # Context manager (with): on exit it calls shutdown(wait=True), so the block doesn't
        # return until all pending futures finish, and the threads are cleaned up.
        # ThreadPoolExecutor runs calls asynchronously on a pool of at most
        # max_workers threads. It suits I/O-bound work such as network requests.
        with ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks  # concurrency cap
        ) as executor:
            future_to_idea = {
                # submit() schedules one call and immediately returns a Future,
                # which is a placeholder for the result.
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            # map yields results in input order. as_completed yields futures as they finish,
            # which is better for progress bars and saving results as they arrive.
            for future in as_completed(future_to_idea):
                try:
                    # If the call raised, result() re-raises the same exception.
                    # If a timeout expires first, it raises TimeoutError.
                    # Wrap each result() call in its own try/except.
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as exc:
                    print(f"Error generating test case: {exc}")

        file_path = Path("data").resolve() / output_file
        if not file_path.parent.exists():
            file_path.parent.mkdir(parents=True, exist_ok=True)

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(
        self,
        test_case: Mapping[str, Mapping[str, str]],
        output: str,
        extra_criteria: str | None,
    ) -> dict[str, str]:
        """Grade the output of a test case using the model"""
        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements.
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies
          in secondary criteria.
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with
          minor issues.
        * Score 9-10: Solution meets all mandatory and secondary criteria.

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own
          extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10.
        * Don't complain that the solution "only" meets the mandatory and secondary criteria.
          Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower.
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted.

        Output Format
        Provide your evaluation as a structured JSON object with the following fields,
        in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages: list[MessageParam] = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
        )
        return json.loads(eval_text)

    def run_test_case(
        self,
        test_case: Mapping[str, Mapping[str, str]],
        run_prompt_function: Callable[[Mapping[str, str]], str],
        extra_criteria: str | None = None,
    ) -> dict[str, Any]:
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_promp_function: Callable[[Mapping[str, str]], str],
        data_set_file: str,
        extra_criteria: str | None = None,
        json_output_file: str = "output.json",
        html_output_file: str = "output.html",
    ) -> list[dict[str, Any]]:
        """Run evaluation on all test cases in the dataset"""
        data_file = Path("data").resolve() / data_set_file
        with open(data_file, encoding="utf-8") as f:
            dataset = json.load(f)

        results: list[dict[str, Any]] = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_promp_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        output_dir = Path("data").resolve() / "output"
        if not output_dir.exists():
            output_dir.mkdir(parents=True, exist_ok=True)

        json_file = output_dir / json_output_file
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        html_file = output_dir / html_output_file
        html_file.write_text(html, encoding="utf-8")

        return results

---
## CREATE INSTANCE OF PromptEvaluator
---
#### Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!

In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)

In [ ]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Write a compact, consise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're
    # getting rate limit errors)
    num_cases=5,
)

---
## DEFINE AND RUN THE PROMPT
---
#### The prompt you want to evaluate, returning the raw model output
#### This function is executed once for each test case

In [ ]:
def run_prompt(prompt_inputs: Mapping[str, str]) -> str:
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Here is an example with a sample input and an ideal output:

    <sample_input>
    height: 165
    weight: 95
    goal: muscle gain
    restrictions: vegan
    </sample_input>

    <ideal_output>
    # One-Day Vegan Muscle-Gain Meal Plan

    **Athlete Profile:** 165 cm, 95 kg | Goal: Muscle Gain | Vegan Diet

    ---

    ## Daily Nutritional Targets
    - **Calories:** 2,800 kcal
    - **Protein:** 160g (22% of calories)
    - **Carbohydrates:** 350g (50% of calories)
    - **Fat:** 78g (25% of calories)

    *Based on ~29.4 BMI; protein prioritized for muscle synthesis at 1.7g/kg body weight*

    ---

    ## MEAL PLAN

    ### **BREAKFAST** (7:00 AM)
    *680 calories | 24g protein | 85g carbs | 18g fat*

    - Oatmeal (dry): 80g
    - Soy milk (unsweetened): 250ml
    - Banana: 120g
    - Almond butter: 32g
    - Flax seeds: 15g

    ---

    ### **MID-MORNING SNACK** (10:30 AM)
    *320 calories | 20g protein | 42g carbs | 8g fat*

    - Vegan protein powder (pea/hemp): 30g
    - Whole grain bread: 60g
    - Peanut butter: 20g
    - Apple: 180g

    ---

    ### **LUNCH** (1:00 PM)
    *750 calories | 38g protein | 92g carbs | 20g fat*

    - Cooked lentils: 200g
    - Brown rice: 150g (cooked)
    - Olive oil: 15ml
    - Spinach: 100g
    - Garlic & cumin seasoning

    ---

    ### **PRE-WORKOUT SNACK** (4:00 PM)
    *290 calories | 12g protein | 48g carbs | 4g fat*

    - Banana: 100g
    - Dates: 50g
    - Hemp seeds: 20g

    ---

    ### **DINNER** (7:00 PM)
    *680 calories | 60g protein | 70g carbs | 24g fat*

    - Tofu (extra-firm, baked): 300g
    - Quinoa: 100g (cooked)
    - Sweet potato: 180g
    - Broccoli: 150g
    - Tahini dressing: 20ml

    ---

    ### **EVENING SNACK** (9:30 PM)
    *80 calories | 6g protein | 13g carbs | 2g fat*

    - Cashew milk: 240ml
    - Nutritional yeast: 10g

    ---

    ## Daily Totals
    ✅ **Calories:** 2,800
    ✅ **Protein:** 160g
    ✅ **Carbs:** 350g
    ✅ **Fat:** 78g

    ---

    ## Key Notes for Muscle Gain
    - **Protein distribution:** Spread throughout day for optimal muscle protein synthesis
    - **Post-workout window:** Consume lunch or pre-workout snack within 2 hours of training
    - **Hydration:** Drink 3-4L water daily
    - **Supplementation:** Consider vegan B12, vitamin D, and omega-3 (algae-based)
    </ideal_output>

    This example meal plan is well-structured, nutritionally sound, and directly addresses the task.
    The meal plan is compact, practical, and evidence-based. It also comprehensively satisfies all
    mandatory requirements with clear presentation of daily totals, macro breakdown, and 6
    well-timed meals with precise portions.
    """

    messages: list[MessageParam] = []
    add_user_message(messages, prompt)
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_promp_function=run_prompt,
    data_set_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
)

---
### PROMPT ENGINEERING TECHNIQUES
---

Original Prompt: What should this person eat?
- Score: 2.6

### 1.- Be Clear And Direct
When crafting that crucial first line, you want to focus on two key principles: clarity and directness. This means using simple language that leaves no room for ambiguity about what you want Claude to do.

Clear Communication -> Instead of writing something vague like "I need to know about those things people put on their roofs that use sun - those solar panel things, I think they're called," be direct and write: "Write three paragraphs about how solar panels work."

Direct Instructions -> Rather than asking "I was reading about renewable energy and geothermal energy sounds neat. What countries use it?" try: "Identify three countries that use geothermal energy. Include generation stats for each."

Try: Generate a one-day meal plan for an athlete that meets their dietary restrictions.
    - Score: 7.8


### 2.- Be Specific

When working with Claude, one of the most effective ways to improve your results is to be specific about what you want. Instead of leaving everything up to the model's interpretation, you can provide clear guidelines or steps that direct Claude toward the kind of output you're looking for.

#### Two Types of Guidelines:

- List Qualities that the output should have.
- Provide steps the model should follow.


Try: 

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts  
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned.

Score: 6.6



### 3.- Provide Structure

- Use XML tags to separate distinct portions of the prompt.
- Most useful when including a lot of different content. 
- They sever as delimiters for Claude.

- Examples:
```xml
<athlete_information>
</athlete_information>
```

Score: 6.2

### 4.- Provide Examples (One-shot or Multi-shot prompting)

- Find responses that scored 10 (or your highest available score) and use those input/output pairs as examples in your prompt. This helps Claude understand what "perfect" output looks like for your specific use case.
- Don't just provide the input/output pair - explain why the output is good.

Score: 8.8